# Suite PLN — Plain

Conversion between values and plain data: the exact snapshot format, symbol assignment, single-object vs reachable
snapshots, deserialization through injected builders, and rejection of malformed snapshots with precise errors.
Plain data keeps ints (`bigint`) apart from floats (`number`) and uses `Map` for mappings.

In [ ]:
import { Plain, Proxies, Reachable, Schemas as S, Validators } from "../src/Framework/index.js";
import { KeyError, NotImplementedError, ValueError } from "../src/Framework/Errors.js";
import type { PlainData, PlainMap } from "../src/Framework/Plain.js";
import { assert, copy, entries, equal, Fake, map, raises, same, same_graph, text } from "./support.js";

let Tag = new S.OfObject.Builder().properties(text("label"), text("n", BigInt), text("x", Number), text("ok", Boolean),
  text("raw", Uint8Array)).create();
const Edge = new S.OfRelation.Builder().links("src", "dst").properties(text("w", BigInt), text("note")).create();
Tag = new S.OfObject.Builder(Tag).relations((r) => r.name("out").of(Edge).me("src"), (r) => r.name("in").of(Edge).me("dst")).update();
Proxies.register("Tag", Tag);
Proxies.register("Edge", Edge);
const B = Proxies.Builders;
const FromPlain = Plain.FromPlain(B);
const objectsOf = (g: PlainData) => (g as PlainMap).get("objects") as Map<string, PlainMap>;

## PLN-01 · Natives: per-kind and dispatching entry points agree

In [ ]:
for (const [schema, value, plain] of [[new S.OfNative.Data(BigInt), 7n, 7n], [new S.OfNative.Data(Uint8Array), new Uint8Array([1]), "AQ=="],
  [new S.OfNative.Data(Number), Infinity, "Infinity"], [new S.OfNative.Data(Boolean), false, false]] as const) {
  assert(equal(Plain.ToPlain(schema, value), plain) && equal(Plain.ToPlain.OfNative(schema, value as never), plain));
  assert(equal(FromPlain(schema, plain), value) && equal(FromPlain.OfNative(schema, plain), value));
}
raises(NotImplementedError, () => Plain.ToPlain(new S.OfUnion.Data(), 1n));
raises(NotImplementedError, () => FromPlain(new S.OfIntersection.Data(), new Map()));
raises(NotImplementedError, () => Plain.ToPlain(Edge, null)); // relations are not serialized on their own

## PLN-02 · The exact snapshot format

In [ ]:
const a = B.Tag().label("a").n(1n).create();
const b = B.Tag().label("b").out((e: any) => e.dst(a).w(5n)).create();
assert(equal(Plain.ToPlain(Tag, b), map({
  root: "s0",
  objects: { s0: { label: "b", out: [{ dst: { $ref: "s1", $schema: "Tag" }, w: 5n }] } },
})));
assert(equal(Plain.ToPlain.Reachable(Tag, b), map({
  root: "s0",
  objects: {
    s0: { label: "b", out: [{ dst: { $ref: "s1", $schema: "Tag" }, w: 5n }] },
    s1: { label: "a", n: 1n, in: [{ src: { $ref: "s0", $schema: "Tag" }, w: 5n }] },
  },
})));
assert(equal(Plain.ToPlain(Tag, B.Tag().create()), map({ root: "s0", objects: { s0: {} } }))); // absent means omitted
{
  const out = objectsOf(Plain.ToPlain(Tag, b)).get("s0") as PlainMap;
  assert(typeof ((out.get("out") as PlainMap[])[0] as PlainMap).get("w") === "bigint"); // ints stay ints
}

## PLN-03 · Properties are written in schema order, entries in relation order

In [ ]:
{
  const t = B.Tag().raw(new Uint8Array()).ok(true).x(0.5).n(0n).label("").create();
  assert(same([...(objectsOf(Plain.ToPlain(Tag, t)).get("s0") as PlainMap).keys()], ["label", "n", "x", "ok", "raw"]));
  const e = B.Tag().label("e").out((x: any) => x.note("n").w(1n).dst(a)).create();
  const first = ((objectsOf(Plain.ToPlain(Tag, e)).get("s0") as PlainMap).get("out") as PlainMap[])[0] as PlainMap;
  assert(same([...first.keys()], ["dst", "w", "note"]));
}

## PLN-04 · Symbols follow first-reference order; the root is always s0

In [ ]:
const hub = B.Tag().label("hub").out((x: any) => x.dst(B.Tag().label("c1").create())).out(
  (x: any) => x.dst(B.Tag().label("c2").create())).create();
const graph = Plain.ToPlain.Reachable(Tag, hub);
assert(graph.get("root") === "s0");
assert(same(["s0", "s1", "s2"].map((s) => (objectsOf(graph).get(s) as PlainMap).get("label")), ["hub", "c1", "c2"]));
assert(same(Reachable.of(hub).map((o: any) => (o instanceof Fake ? o.values.get("label") : o.label)), ["hub", "c1", "c2"]));

## PLN-05 · Single-object snapshots leave references unresolved; FromPlain rejects them

In [ ]:
{
  const single = Plain.ToPlain(Tag, hub);
  assert(same([...objectsOf(single).keys()], ["s0"]));
  raises(ValueError, () => FromPlain(Tag, single), "unresolved reference 's1'");
  assert((FromPlain(Tag, Plain.ToPlain(Tag, B.Tag().label("alone").create())) as any).label === "alone");
}

## PLN-06 · Round trips build new objects; entries seen from both ends are not duplicated

In [ ]:
{
  const back = FromPlain.Reachable(Tag, graph) as Proxies.Instance;
  assert(back !== hub && back.label === "hub");
  assert(same_graph(Plain.ToPlain.Reachable(Tag, back), graph));
  assert(entries(Tag, back, "out").length === 2);
  assert(Validators.Validate(B).Reachable(Tag, back).length === 0);
  const lone = B.Tag().label("lone").n(3n).create();
  const loneBack = FromPlain.OfObject(Tag, Plain.ToPlain.OfObject(Tag, lone)) as Proxies.Instance;
  assert(loneBack !== lone && loneBack.label === "lone" && loneBack.n === 3n);
}

## PLN-07 · Values at their edges survive a round trip

In [ ]:
{
  const all = new Uint8Array(Array.from({ length: 256 }, (_, i) => i));
  const edge = B.Tag().label("\x00 日本語 🚀 \ud800").n(-(2n ** 100n)).x(-0.0).ok(false).raw(all).create();
  const edgeBack = FromPlain(Tag, Plain.ToPlain(Tag, edge)) as Proxies.Instance;
  assert(edgeBack.label === edge.label && edgeBack.n === edge.n && edgeBack.ok === false && equal(edgeBack.raw, edge.raw));
  assert(Object.is(edgeBack.x, -0));
  for (const special of [NaN, Infinity, -Infinity]) {
    const got = (FromPlain(Tag, Plain.ToPlain(Tag, B.Tag().x(special).create())) as Proxies.Instance).x;
    assert((Number.isNaN(got) && Number.isNaN(special)) || got === special);
  }
}

## PLN-08 · Finding F6 (pinned): a round trip can renumber symbols
Restoring re-adds each object's entries, which moves them to the end of the relation's table, so entry order and
therefore first-reference numbering can change. Content is preserved (compare up to renumbering).

In [ ]:
{
  // The smallest case (found by search in Python): n1 links to itself, then to n0; the snapshot is rooted at n0.
  const n0 = B.Tag().label("n0").create();
  const n1 = B.Tag().label("n1").create();
  B.Tag(n1).out((x: any) => x.dst(n1)).update();
  B.Tag(n1).out((x: any) => x.dst(n0)).update();
  const before = Plain.ToPlain.Reachable(Tag, n0);
  const after = Plain.ToPlain.Reachable(Tag, FromPlain.Reachable(Tag, before) as Proxies.Instance);
  assert(same_graph(before, after));
  assert(!equal(before, after)); // exact equality does not hold: n1's entries come back in a different order
}

## PLN-09 · ToPlain checks the root schema and native types

In [ ]:
raises(TypeError, () => Plain.ToPlain(new S.OfObject.Data(), a), "not an instance of the given schema");
raises(TypeError, () => Plain.ToPlain(Tag, B.Tag().n("1").create()), "expected int, got str");
raises(TypeError, () => Plain.ToPlain.Reachable(Tag, B.Tag().out((x: any) => x.dst(B.Tag().n(true).create())).create())); // also when reached

## PLN-10 · Hand-written visitables serialize too

In [ ]:
{
  const fa = new Fake("Tag", { label: "fa" });
  const fb = new Fake("Tag", { label: "fb" }, { out: [{ dst: fa, w: 1n }] });
  const fg = Plain.ToPlain.Reachable(Tag, fb);
  assert(equal(objectsOf(fg).get("s0"), map({ label: "fb", out: [{ dst: { $ref: "s1", $schema: "Tag" }, w: 1n }] })));
  assert(equal(objectsOf(fg).get("s1"), map({ label: "fa" }))); // a fake only writes what it holds: no mirrored 'in' entry
  const restored = FromPlain.Reachable(Tag, fg) as Proxies.Instance;
  assert(same(entries(Tag, restored, "out").map((x) => x.get("dst").label), ["fa"]));
  raises(KeyError, () => Plain.ToPlain(Tag, new Fake("Tag", { colour: "red" })), "unknown property");
  raises(KeyError, () => Plain.ToPlain(Tag, new Fake("Tag", {}, { sideways: [{}] })), "unknown adjacency");
}

## PLN-11 · Malformed snapshots are rejected before anything is built, with precise errors

In [ ]:
const ga = B.Tag().label("ga").create();
const gb = B.Tag().label("gb").out((x: any) => x.dst(ga).w(5n)).create();
const good = Plain.ToPlain.Reachable(Tag, gb); // exactly two objects, each referenced once

function mutated(change: (s: PlainMap) => unknown): PlainMap {
  const snapshot = copy(good);
  change(snapshot);
  return snapshot;
}
const obj = (s: PlainMap, symbol: string) => objectsOf(s).get(symbol) as PlainMap;
const link0 = (s: PlainMap) => (obj(s, "s0").get("out") as PlainMap[])[0] as PlainMap;
const ref0 = (s: PlainMap) => link0(s).get("dst") as PlainMap;
const inRef = (s: PlainMap) => ((obj(s, "s1").get("in") as PlainMap[])[0] as PlainMap).get("src") as PlainMap;

const cases: [PlainData, string][] = [
  [[], "expected an object snapshot"],
  [map({ root: "s0" }), "expected an object snapshot"],
  [map({ root: "s0", objects: {}, extra: 1n }), "expected an object snapshot"],
  [map({ root: "s0", objects: [] }), "expected an object snapshot"],
  [map({ root: "zz", objects: { s0: {} } }), "root 'zz' is not in the snapshot"],
  [map({ root: "s0", objects: { s0: [] } }), "must be a mapping"],
  [mutated((s) => obj(s, "s0").set("out", "oops")), "must be a list of entries"],
  [mutated((s) => (obj(s, "s0").get("out") as PlainData[]).push("oops")), "an entry must be a mapping"],
  [mutated((s) => ref0(s).delete("$schema")), "a reference is"],
  [mutated((s) => ref0(s).set("extra", 1n)), "a reference is"],
  [mutated((s) => ref0(s).set("$ref", 1n)), "a reference is"],
  [mutated((s) => ref0(s).set("$schema", "Nope")), "no object schema registered as 'Nope'"],
  [mutated((s) => ref0(s).set("$schema", "Edge")), "no object schema registered as 'Edge'"],
  [mutated((s) => ref0(s).set("$ref", "s9")), "unresolved reference 's9'"],
  [mutated((s) => link0(s).set("dst", "s1")), "a link must be a reference"],
  [mutated((s) => link0(s).delete("dst")), "links ['dst'] are not set"],
  [mutated((s) => link0(s).set("src", map({ $ref: "s0", $schema: "Tag" }))), "own link, which is implied"],
  [mutated((s) => link0(s).set("colour", "red")), "no link or property 'colour'"],
  [mutated((s) => link0(s).set("w", [1n])), "a property must be a plain value"],
  [mutated((s) => obj(s, "s0").set("label", map({ x: 1n }))), "a property must be a plain value"],
  [mutated((s) => obj(s, "s0").set("colour", "red")), "has no property or adjacency 'colour'"],
  [mutated((s) => objectsOf(s).set("s9", new Map())), "nothing references: ['s9']"],
  [mutated((s) => inRef(s).set("$schema", "Other")), "referenced as both"],
];
for (const [snapshot, message] of cases) raises(ValueError, () => FromPlain.Reachable(Tag, snapshot), message);

## PLN-11b · Ends that disagree are not malformed: each end's entries are restored
Each entry is written under every object it links. If the two ends of a snapshot disagree, both versions are
restored (the union of entries), since each is well-formed on its own. Validation is the place to catch this.

In [ ]:
{
  const asymmetric = mutated((s) => inRef(s).set("$ref", "s1"));
  const restoredA = FromPlain.Reachable(Tag, asymmetric) as Proxies.Instance;
  assert(same(entries(Tag, restoredA, "out").map((x) => x.get("dst").label), ["ga"]));
  const gaBack = entries(Tag, restoredA, "out")[0]?.get("dst");
  assert(same(entries(Tag, gaBack, "in").map((x) => x.get("src").label).sort(), ["ga", "gb"])); // s0 -> s1 and s1 -> s1
}

## PLN-12 · Type errors in values surface as `TypeError`; a bad root schema before parsing

In [ ]:
raises(TypeError, () => FromPlain.Reachable(Tag, mutated((s) => link0(s).set("w", "5"))), "expected int, got str");
raises(TypeError, () => FromPlain.Reachable(Tag, mutated((s) => obj(s, "s0").set("label", 5n))), "expected str, got int");
raises(TypeError, () => FromPlain.Reachable(Edge as never, good), "root schema must be an object schema");
raises(TypeError, () => FromPlain.Reachable(new S.OfNative.Data(String) as never, good), "root schema must be an object schema");

## PLN-13 · A snapshot that is rejected builds nothing and leaves the input untouched

In [ ]:
{
  const bad = mutated((s) => link0(s).delete("dst"));
  const frozen = copy(bad);
  const beforeIds = new Set(Reachable.of(gb).map((o) => o.identity()));
  raises(ValueError, () => FromPlain.Reachable(Tag, bad));
  assert(equal(bad, frozen));
  assert(same([...new Set(Reachable.of(gb).map((o) => o.identity()))].sort(), [...beforeIds].sort()));
}

## PLN-14 · FromPlain only uses the builders it is given

In [ ]:
{
  const names: string[] = [];
  const recording = new Proxy(B, {
    get(target, prop, receiver) {
      if (prop !== "schema" && prop !== "name_of" && typeof prop === "string") names.push(prop);
      return Reflect.get(target, prop, receiver);
    },
  });
  Plain.FromPlain(recording).Reachable(Tag, good);
  assert(names.length > 0 && same([...new Set(names)], ["Tag"]));
  const failing = new Proxy(B, {
    get(target, prop, receiver) {
      if (prop === "schema") return () => { throw new RangeError("registry down"); };
      return Reflect.get(target, prop, receiver);
    },
  });
  raises(RangeError, () => Plain.FromPlain(failing).Reachable(Tag, good), "registry down"); // only lookup failures are rewritten
}

## PLN-15 · Property schemas that are not native are refused; per-kind text entry points

In [ ]:
import { JSON, YAML } from "../src/Framework/index.js";
{
  const Holder = new S.OfObject.Builder().properties((p) => p.name("inner").of(Tag), text("label")).create();
  Proxies.register("Holder", Holder);
  const holder = B.Holder().inner("anything").create(); // the builder does not check (F8)
  raises(TypeError, () => Plain.ToPlain(Holder, holder), "property 'inner' is not native");
  raises(NotImplementedError, () => FromPlain(Holder, map({ root: "s0", objects: { s0: { inner: "x" } } })),
    "property 'inner': only native values are supported by Plain yet");
  raises(NotImplementedError, () => Plain.ToPlain("not a schema", 1n), "str is not supported by Plain yet");
  raises(NotImplementedError, () => FromPlain("not a schema", 1n), "str is not supported by Plain yet");

  const lone = B.Tag().label("lone-text").n(9n).create();
  assert((JSON.FromJSON(B).OfObject(Tag, JSON.ToJSON.OfObject(Tag, lone)) as any).n === 9n);
  assert((YAML.FromYAML(B).OfObject(Tag, YAML.ToYAML.OfObject(Tag, lone)) as any).n === 9n);
}